# 01 - Exploratory data analysis

**Goal of this notebook:** understand the raw data before building any model.
We check that train and test files are consistent, look at missing values, study the
four targets (`M`, `MM`, `Combo`, `DCombo`) and detect data-quality problems that the
cleaning step (notebook 02) will have to solve.

Every decision taken later in the project is justified by a finding in this notebook.

## 1. Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from miopia.config import FIGURES, RANDOM_STATE
from miopia.data import load_raw

# Display options: show all columns and round floats for readability
pd.set_option("display.max_columns", 60)
pd.set_option("display.precision", 2)
sns.set_theme(style="whitegrid", context="notebook")
FIGURES.mkdir(parents=True, exist_ok=True)

X_train, Y_train, X_test = load_raw()
print("X_train:", X_train.shape, "| Y_train:", Y_train.shape, "| X_test:", X_test.shape)

In [ ]:
# First rows of the features and the labels
display(X_train.head())
display(Y_train.head())

## 2. Are train and test consistent?

Both feature files must have the same columns, in the same order and with the same types,
otherwise a model trained on one cannot be applied to the other.

In [ ]:
# Same column names and order?
print("Same columns and order:", list(X_train.columns) == list(X_test.columns))

# Columns whose dtype differs between train and test
dtype_diff = pd.DataFrame({"train": X_train.dtypes, "test": X_test.dtypes})
dtype_diff[dtype_diff["train"] != dtype_diff["test"]]

The column names match. Three numeric columns have a different dtype (`int64` in one file,
`float64` in the other). This is only a storage format issue: pandas reads a column as
float when it contains a decimal value or a missing value (`NaN` only exists for floats).
We cast every numeric column to `float` in both files so they have exactly the same schema.

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns
X_train[num_cols] = X_train[num_cols].astype(float)
X_test[num_cols] = X_test[num_cols].astype(float)

print("All dtypes match now:", (X_train.dtypes == X_test.dtypes).all())
print(f"{len(num_cols)} numeric columns, {X_train.shape[1] - len(num_cols)} text columns")

## 3. Missing values

In [ ]:
# Percentage of missing values per column, in train and test
missing = pd.DataFrame({
    "train_%": X_train.isna().mean() * 100,
    "test_%": X_test.isna().mean() * 100,
}).round(1)
missing = missing[missing.sum(axis=1) > 0].sort_values("train_%", ascending=False)
missing

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing.plot.barh(ax=ax)
ax.set_xlabel("% missing")
ax.set_title("Missing values per column")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES / "01_missing_values.png", dpi=150)
plt.show()

- `hº act.cerca sem` is **100 % empty** in both files: it carries no information and will be dropped.
- `origen antepasados (extranjeros)` is ~75 % empty, but here a blank most likely means
  "no foreign ancestors", not "unknown".
- `fototipo` / `Grupo fot` have ~17 % missing values and will need imputation.
- The rest have only a few missing values (≤ 8 %).

## 4. Targets

The professor provides four targets. Note two differences with the brief:
labels are written `SI`/`NO` (not `YES`/`NO`), and `Combo`/`DCombo` are **multi-class**
problems (one label per student), not multi-label.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, Y_train.columns):
    counts = Y_train[col].value_counts().sort_index()
    counts.plot.bar(ax=ax, color=sns.color_palette("Set2", len(counts)))
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=0)
    for i, v in enumerate(counts):
        ax.text(i, v + 1, str(v), ha="center")
plt.suptitle("Class distribution of the four targets (n = 132)", y=1.03)
plt.tight_layout()
plt.savefig(FIGURES / "01_targets.png", dpi=150, bbox_inches="tight")
plt.show()

**Class imbalance.** `MM` is the hardest task: only **10 positives out of 132** (7.6 %).
A model that always predicts `NO` would reach 92 % accuracy while being useless, so
accuracy alone is not a valid metric. We will use F1, balanced accuracy and stratified
cross-validation.

### 4.1 Are the four targets consistent with each other?

In [ ]:
# Relationship between the targets
display(pd.crosstab(Y_train["Combo"], Y_train["MM"], margins=True))
display(pd.crosstab(Y_train["Combo"], Y_train["DCombo"]))

In [ ]:
# Students labelled as myopia magna in Combo but NOT in the MM column
inconsistent = (Y_train["Combo"] == "MM") & (Y_train["MM"] == "NO")
print("Inconsistent rows:", inconsistent.sum())
pd.concat([X_train.loc[inconsistent, ["LA OD", "LA OS", "LA Media", "familiar miope"]],
           Y_train.loc[inconsistent]], axis=1)

**Label inconsistency found.** 8 students have `Combo = MM` but `MM = NO`.
`DCombo` agrees with `Combo` for them (`MM`), and `M` is `SI` for all of them.
The next cell compares their axial length (the main anatomical marker of myopia) with
the other groups.

In [ ]:
groups = np.select(
    [Y_train["Combo"] == "C", Y_train["Combo"] == "M", inconsistent],
    ["C", "M", "MM (Combo only)"], default="MM (both)")

fig, ax = plt.subplots(figsize=(7, 4))
order = ["C", "M", "MM (Combo only)", "MM (both)"]
sns.boxplot(x=groups, y=X_train["LA Media"], order=order, ax=ax, palette="Set2", hue=groups, legend=False)
sns.stripplot(x=groups, y=X_train["LA Media"], order=order, ax=ax, color="k", size=3)
ax.set_xlabel("")
ax.set_ylabel("Mean axial length (mm)")
ax.set_title("Axial length by label group")
plt.tight_layout()
plt.savefig(FIGURES / "01_inconsistent_labels.png", dpi=150)
plt.show()

pd.Series(X_train["LA Media"].values, index=groups).groupby(level=0).describe()[["count", "mean", "min", "max"]]

The 8 "Combo-only" students have the same axial length as the confirmed myopia-magna
students (~26.5 mm) and much larger than simple myopes (~24.5 mm). This suggests they are
myopia-magna cases mislabelled in the `MM` column (or that both columns were built with
different criteria).

**Decision (to be confirmed by the team):** each task is trained on its own column as
provided, because the test set is most likely graded that way. We also keep this finding
for the report and test, as an extra experiment, the effect of correcting these 8 labels.

## 5. Data-quality issues in the features

### 5.1 `fototipo` and `Grupo fot`

In [ ]:
pd.crosstab(X_train["fototipo"].fillna("NaN"), X_train["Grupo fot"].fillna("NaN"))

`fototipo` should contain the skin phototype (1 to 5), but 38 rows contain `E` or `L`,
which are the values of `Grupo fot`. The two variables were mixed during data entry.
`Grupo fot` is almost constant (only 2 students are `L`), so it is nearly useless.
We will extract the numeric phototype (1-5) and treat `E`/`L` as missing.

### 5.2 Exam date

In [ ]:
X_train["fecha"].value_counts()

All exams took place in February-March 2020, except 4 rows with the year `97`, clearly a typing
error. The exam date has no causal relation with myopia, so `fecha` will be dropped.

### 5.3 Derived (redundant) columns

In [ ]:
# Several columns are sums or means of others. We check it numerically.
outdoor_sum = X_train[["hº deporte sem", "hº naturaleza sem", "hº ocio exteriores sem"]].sum(axis=1)  # missing parts count as 0
checks = {
    "horas exterior = deporte + naturaleza + ocio ext.":
        np.allclose(X_train["horas exterior sem"], outdoor_sum, equal_nan=True),
    "LA Media = (LA OD + LA OS) / 2":
        np.allclose(X_train["LA Media"], (X_train["LA OD"] + X_train["LA OS"]) / 2),
    "CUVAF-Total = OD-Total + OS-Total":
        np.allclose(X_train["CUVAF-Total"], X_train["OD-Total"] + X_train["OS-Total"]),
    "Suma Nasal = 2 * Promedio Nasal":
        np.allclose(X_train["Suma Nasal"], 2 * X_train["Promedio Nasal"]),
}
pd.Series(checks, name="holds")

In [ ]:
# Correlation between numeric features (absolute value)
corr = X_train[num_cols].drop(columns="hº act.cerca sem").corr().abs()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="Blues", square=True, cbar_kws={"shrink": .6}, ax=ax)
ax.set_title("Absolute correlation between numeric features")
plt.tight_layout()
plt.savefig(FIGURES / "01_correlation.png", dpi=150)
plt.show()

Many features are exact combinations of others (right eye / left eye / mean / total).
Keeping all of them adds collinearity without new information, which hurts linear models
and makes feature importances hard to interpret. In feature engineering we will keep one
representative per group (e.g. the mean of both eyes).

### 5.4 Free-text and categorical columns

In [ ]:
cat_cols = X_train.columns.difference(num_cols)
summary = pd.DataFrame({
    "n_unique": [pd.concat([X_train[c], X_test[c]]).nunique() for c in cat_cols],
    "examples": [pd.concat([X_train[c], X_test[c]]).dropna().unique()[:4].tolist() for c in cat_cols],
}, index=cat_cols).sort_values("n_unique", ascending=False)
summary

- `origen antepasados (españa)` has 95 different free-text values for 156 students: too
  sparse to be used directly. A simple derived flag (has foreign ancestors yes/no) is more
  useful.
- `familiar miope` lists which relatives are myopic (father, mother, siblings). It can be
  turned into binary columns (`father_myopic`, `mother_myopic`, `siblings_myopic`).
- The remaining columns are small yes/no or ordinal categories (`Nunca` < `A veces` < `Siempre`).

## 6. Which features are related to myopia?

In [ ]:
# Distribution of the main numeric features for myopic (SI) vs non-myopic (NO) students
key_vars = ["LA Media", "Querato OD", "horas exterior sem", "hº estudiar sem",
            "horas interior sem", "Ratio Cerca/Lejos", "CUVAF-Total", "edad"]
df = X_train.join(Y_train)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, var in zip(axes.ravel(), key_vars):
    sns.boxplot(data=df, x="Combo", y=var, order=["C", "M", "MM"], ax=ax,
                palette="Set2", hue="Combo", legend=False)
    ax.set_xlabel("")
plt.suptitle("Key numeric features by Combo class", y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / "01_numeric_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Non-parametric test (Kruskal-Wallis) to rank numeric features by how well they separate C / M / MM
from scipy.stats import kruskal

rows = []
for var in num_cols.drop("hº act.cerca sem"):
    samples = [df.loc[df["Combo"] == g, var].dropna() for g in ["C", "M", "MM"]]
    stat, p = kruskal(*samples)
    rows.append((var, stat, p))
ranking = pd.DataFrame(rows, columns=["feature", "H", "p_value"]).sort_values("p_value")
ranking.head(12)

Axial length (`LA`) is by far the strongest predictor (p ~ 1e-14), followed at a long
distance by the number of myopic parents and the weekly hours of sport: it grows from ~23.5 mm (controls)
to ~24.5 mm (myopia) and ~26.4 mm (myopia magna). This is consistent with the medical
literature: myopia is mainly caused by an elongated eyeball.

In [ ]:
# Proportion of myopic students for each value of some categorical features
cat_vars = ["sexo", "familiar miope", "Familiar MM SI/NO", "disp.elec.antes dormir", "Exterior Mucho", "toma sol"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, var in zip(axes.ravel(), cat_vars):
    prop = pd.crosstab(df[var].fillna("NaN"), df["M"], normalize="index")["SI"]
    n = df[var].fillna("NaN").value_counts()
    prop.sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_xlim(0, 1)
    ax.set_xlabel("% with myopia (M = SI)")
    ax.set_ylabel("")
    ax.set_title(var)
    for i, k in enumerate(prop.sort_values().index):
        ax.text(0.01, i, f"n={n[k]}", va="center", color="white", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / "01_categorical_vs_M.png", dpi=150)
plt.show()

## 7. Do train and test come from the same distribution?

If the test students were very different from the training students, the validation score
would be an optimistic estimate of the test score.

In [ ]:
from scipy.stats import ks_2samp

# Kolmogorov-Smirnov test on each numeric column (small p-value = different distributions)
shift = []
for var in num_cols.drop("hº act.cerca sem"):
    stat, p = ks_2samp(X_train[var].dropna(), X_test[var].dropna())
    shift.append((var, round(stat, 2), round(p, 3)))
shift = pd.DataFrame(shift, columns=["feature", "KS", "p_value"]).sort_values("p_value")
shift.head(8)

With only 24 test students the test has little power, but no feature shows a strong shift,
so train and test can be treated as samples from the same population.

## 8. Summary of findings and decisions for notebook 02

| # | Finding | Action |
|---|---|---|
| 1 | Same columns; 3 numeric columns with different dtype (int/float) | Cast all numeric columns to float |
| 2 | `hº act.cerca sem` 100 % empty | Drop |
| 3 | `fecha` irrelevant and with typos | Drop |
| 4 | `fototipo` mixed with `Grupo fot` values (`E`, `L`) | Keep numeric phototype 1-5, `E`/`L` → missing, impute |
| 5 | `Grupo fot` almost constant | Drop |
| 6 | Many redundant columns (eye OD/OS/mean/total, outdoor sum) | Keep one representative per group |
| 7 | Free-text ancestry (95 values) | Binary flag "foreign ancestors" |
| 8 | `familiar miope` lists relatives | One binary column per relative |
| 9 | 8 rows with `Combo = MM` but `MM = NO` | Train each task on its own column; test correction as extra experiment |
| 10 | Strong imbalance (`MM`: 10 / 132) | Stratified CV, F1 / balanced accuracy, class weights |
| 11 | Axial length is the dominant predictor | Expect it at the top of feature importances |